# Olist Brazilian E-Commerce Analysis

Imports and Engine Set-up

In [ ]:
from pathlib import Path
import os

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import URL, create_engine
from dotenv import load_dotenv

load_dotenv()

url_object = URL.create(
    "postgresql+psycopg",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host="localhost",
    port=5433,
    database=os.getenv("DB_NAME"),
)

engine = create_engine(url_object)

FIGURES_PATH = Path("../figures/")
FIGURES_PATH.mkdir(parents=True, exist_ok=True)

### Question 1: Which Order categoriy drive the most revenue?

In [ ]:
Q1_REVENUE_BY_CATEGORY = """
    SELECT
        SUM(price) AS total_revenue,
        product_category_name_english AS category
    FROM
        order_items
    INNER JOIN products
        ON order_items.product_id = products.product_id
    INNER JOIN category_name_translation
        ON products.product_category_name = category_name_translation.product_category_name
    GROUP BY product_category_name_english
    ORDER BY total_revenue DESC
    LIMIT 10;
"""

with engine.connect() as conn:
    df_revenue = pd.read_sql(Q1_REVENUE_BY_CATEGORY, con=conn)

df_revenue.head()

In [ ]:
df_revenue_sorted = df_revenue.sort_values("total_revenue", ascending=True)
df_revenue_sorted["total_revenue"] = df_revenue_sorted.total_revenue / 1000000

plt.barh(df_revenue_sorted.category, df_revenue_sorted.total_revenue)

plt.title("Olist E-Commerce Revenue by Categories")
plt.ylabel("Order Categories")
plt.xlabel("Total Revenue (R$)")

plt.savefig(FIGURES_PATH / "01_orders_revenue_by_categories.png", bbox_inches="tight")
plt.show()

**What it shows:** Health & beauty leads category revenue at R\$1.26M, followed closely by watches/gifts (R\$1.2M) and bed/bath/table (R\$1.04M), while all remaining categories fall below R\$1M.

**What it means:** Olist's customers primarily purchase personal care and home lifestyle products, suggesting the platform's strongest market is everyday consumer goods rather than electronics or industrial categories.

**What to do about it:** Olist should investigate whether underperforming categories suffer from a supply (seller) or demand (buyer) gap, and explore whether top-performing categories can scale further through additional seller recruitment, targeted promotions, or geographic expansion into underserved states.